# Proyecto Final NLP: datos

Descarga del corpus, partición 80/10/10, exploración y vocabularios. Corre en CPU.

In [ ]:
REPO_URL = 'https://github.com/EmilianoRL/NLP_Proyecto.git'
REPO_DIR = 'NLP_Proyecto'

import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('common'):
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    %cd {REPO_DIR}
elif not os.path.exists('common') and os.path.exists('../common'):
    %cd ..

In [ ]:
import random
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

seed = 1234
random.seed(seed)
np.random.seed(seed)

Montamos Drive para guardar los datos y los checkpoints:

In [ ]:
from common import checkpoint as ckpt

SAVE_DIR = ckpt.setup_save_dir()
DATA_DIR = os.path.join(SAVE_DIR, 'data')
FIG_DIR = os.path.join(SAVE_DIR, 'figures')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

## Corpus

Se descarga `spa-eng.zip` y se guarda una copia en Drive, para que los dos usemos siempre el mismo archivo.

In [ ]:
from common.data import download_corpus, file_sha256

corpus_path = download_corpus('.', cache_dir=DATA_DIR)
corpus_sha = file_sha256(corpus_path)

## Partición

Bloque del enunciado, sin cambios:

In [ ]:
import random, hashlib

# 1. Descarga directa en Colab / entorno Linux:
# !wget -q https://www.manythings.org/anki/spa-eng.zip && unzip -q -o spa-eng.zip

with open('spa.txt', 'r', encoding='utf-8') as f:
    raw_pairs = [line.strip().split('\t')[:2] for line in f if '\t' in line]

# 2. Filtrado de complejidad mínima (>= 8 palabras en inglés, <= 120 tokens)
filtered_pairs = [
    (en, es) for en, es in raw_pairs
    if 8 <= len(en.split()) <= 120 and len(es.split()) <= 120
]

# 3. Mezcla determinista con semilla fija (obligatoria para auditoría)
rng = random.Random(23)
rng.shuffle(filtered_pairs)

# 4. Partición estricta 80% Train / 10% Validation / 10% Test
n = len(filtered_pairs)
n_train = int(0.80 * n)
n_val   = int(0.10 * n)

train_pairs = filtered_pairs[:n_train]
val_pairs   = filtered_pairs[n_train:n_train + n_val]
test_pairs  = filtered_pairs[n_train + n_val:]

# 5. Huella de auditoría (debe reportarse en el notebook y en el PDF)
test_hash = hashlib.sha256("\n".join([p[0] for p in test_pairs]).encode()).hexdigest()[:12]
print(f"Train: {len(train_pairs)} | Val: {len(val_pairs)} | Test: {len(test_pairs)}")
print(f"SHA-256 Test Audit ID: {test_hash}")

Revisamos que sea la misma partición que ya teníamos (`dbac04ac5648`) y la guardamos en Drive.
Los demás notebooks la cargan con `get_splits()`.

In [ ]:
from common.data import (load_splits, get_splits, save_splits_json, load_splits_json,
                         EXPECTED_COUNTS, EXPECTED_TEST_AUDIT_ID)

# la función de common/data.py debe dar lo mismo que el bloque de arriba
tr, va, te, h = load_splits('spa.txt', verbose=False)
assert tr == train_pairs and va == val_pairs and te == test_pairs and h == test_hash
assert (len(train_pairs), len(val_pairs), len(test_pairs)) == EXPECTED_COUNTS
assert test_hash == EXPECTED_TEST_AUDIT_ID

splits_path = os.path.join(DATA_DIR, 'splits.json')
if os.path.exists(splits_path):
    tr, va, te, h = load_splits_json(splits_path)
    assert tr == train_pairs and va == val_pairs and te == test_pairs
else:
    save_splits_json(splits_path, train_pairs, val_pairs, test_pairs)

audit = {
    'n_raw_pairs': len(raw_pairs),
    'n_filtered_pairs': n,
    'n_train': len(train_pairs),
    'n_val': len(val_pairs),
    'n_test': len(test_pairs),
    'test_audit_id': test_hash,
    'corpus_sha256': corpus_sha,
}
with open(os.path.join(DATA_DIR, 'audit.json'), 'w') as f:
    json.dump(audit, f, indent=2)

_ = get_splits(DATA_DIR)

## Exploración

In [ ]:
pd.set_option('display.max_colwidth', 120)
pd.DataFrame(train_pairs[:5], columns=['en', 'es'])

In [ ]:
def length_table(pairs, name):
    return pd.DataFrame({
        'split': name,
        'palabras_en': [len(en.split()) for en, es in pairs],
        'palabras_es': [len(es.split()) for en, es in pairs],
    })

lengths = pd.concat([length_table(train_pairs, 'train'),
                     length_table(val_pairs, 'val'),
                     length_table(test_pairs, 'test')])
lengths.groupby('split')[['palabras_en', 'palabras_es']].describe().round(1).T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in zip(axes, ['palabras_en', 'palabras_es'], ['inglés', 'español']):
    for split in ['train', 'val', 'test']:
        ax.hist(lengths.loc[lengths.split == split, col], bins=range(0, 61), alpha=0.5,
                density=True, label=split)
    ax.axvline(18.5, color='black', linestyle='--', linewidth=1)
    ax.set_title(f'Longitud de las oraciones ({title})')
    ax.set_xlabel('palabras')
    ax.set_ylabel('proporción')
    ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'distribucion_longitudes.png'), dpi=150)
plt.show()

Oraciones medias (8 a 18 palabras en inglés) y largas (más de 18), que se usan en la tabla de resultados:

In [ ]:
from common.data import length_bucket

buckets = pd.DataFrame({
    split: pd.Series([length_bucket(en) for en, es in pairs]).value_counts()
    for split, pairs in [('train', train_pairs), ('val', val_pairs), ('test', test_pairs)]
}).reindex(['media', 'larga'])
buckets.loc['% larga'] = (100 * buckets.loc['larga'] / buckets.sum()).round(2)
buckets

Tatoeba tiene varias traducciones para la misma frase en inglés, así que una frase de test puede
aparecer también en train (con otra traducción). Solo lo contamos para tenerlo en cuenta al analizar el BLEU:

In [ ]:
train_en = set(en for en, es in train_pairs)
overlap_val = sum(en in train_en for en, es in val_pairs)
overlap_test = sum(en in train_en for en, es in test_pairs)
pd.DataFrame({
    'pares': [len(val_pairs), len(test_pairs)],
    'inglés repetido en train': [overlap_val, overlap_test],
    '%': [round(100 * overlap_val / len(val_pairs), 2), round(100 * overlap_test / len(test_pairs), 2)],
}, index=['val', 'test'])

## Limpieza

`clean()` pasa a minúsculas y quita acentos y puntuación. Se usa también al calcular BLEU,
sobre las predicciones y las referencias.

In [ ]:
from common.preprocess import clean

for en, es in train_pairs[:3]:
    print(en, '->', clean(en))
    print(es, '->', clean(es))
    print()

## Vocabulario

Se construye solo con train. Probamos varios mínimos de frecuencia y vemos cuántas palabras de val quedarían como `<unk>`:

In [ ]:
from common.vocab import build_vocab, oov_rate

train_en = [clean(en) for en, es in train_pairs]
train_es = [clean(es) for en, es in train_pairs]
val_en = [clean(en) for en, es in val_pairs]
val_es = [clean(es) for en, es in val_pairs]

rows = []
for count_threshold in [1, 2, 3, 5]:
    e2i = build_vocab(train_en, count_threshold)
    s2i = build_vocab(train_es, count_threshold)
    rows.append({
        'count_threshold': count_threshold,
        'vocab_en': len(e2i),
        'vocab_es': len(s2i),
        '% <unk> val (en)': round(oov_rate(val_en, e2i), 2),
        '% <unk> val (es)': round(oov_rate(val_es, s2i), 2),
        '% <unk> train (es)': round(oov_rate(train_es, s2i), 2),
    })
pd.DataFrame(rows).set_index('count_threshold')

Usamos 1, es decir, sin umbral (como en la guía de las slides de Transformer parte 2). Con 2, las palabras que solo
aparecen una vez en train se vuelven `<unk>`, el modelo aprende a generar `<unk>` y eso se notaba mucho en las traducciones
del Modelo A. Con 1 solo aparece `<unk>` para palabras que nunca salen en train.

In [ ]:
from common.vocab import save_vocab, invert, sentence_to_ids, ids_to_sentence

COUNT_THRESHOLD = 1
eng_word2idx = build_vocab(train_en, COUNT_THRESHOLD)
spa_word2idx = build_vocab(train_es, COUNT_THRESHOLD)
eng_idx2word, spa_idx2word = invert(eng_word2idx), invert(spa_word2idx)
print('vocab en:', len(eng_word2idx), '| vocab es:', len(spa_word2idx))
print(list(spa_word2idx.items())[:10])

save_vocab(os.path.join(DATA_DIR, 'vocab.json'), eng_word2idx, spa_word2idx, COUNT_THRESHOLD)

In [ ]:
from common.vocab import UNK_IDX, PAD_TOKEN, UNK_TOKEN, wrap

# todo lo que está en el vocabulario sale de train
train_en_words = set(w for s in train_en for w in wrap(s))
train_es_words = set(w for s in train_es for w in wrap(s))
assert set(eng_word2idx) - {PAD_TOKEN, UNK_TOKEN} <= train_en_words
assert set(spa_word2idx) - {PAD_TOKEN, UNK_TOKEN} <= train_es_words

# ejemplo de una oración de val con una palabra desconocida
for es in val_es:
    ids = sentence_to_ids(es, spa_word2idx)
    if UNK_IDX in ids:
        print(es)
        print(ids)
        print(ids_to_sentence(ids, spa_idx2word))
        break

Largo máximo en tokens (el `collate_fn` corta en `MAX_SEQ_LEN = 128`):

In [ ]:
from common.datasets import pairs_to_ids, MAX_SEQ_LEN

for name, pairs in [('train', train_pairs), ('val', val_pairs)]:
    src, tgt = pairs_to_ids(pairs, eng_word2idx, spa_word2idx)
    n_long = sum(len(s) > MAX_SEQ_LEN or len(t) > MAX_SEQ_LEN for s, t in zip(src, tgt))
    print(f'{name}: max en = {max(map(len, src))}, max es = {max(map(len, tgt))}, más largas que {MAX_SEQ_LEN}: {n_long}')

## Resumen

In [ ]:
summary = {
    **audit,
    'test_media (8-18)': int(buckets.loc['media', 'test']),
    'test_larga (>18)': int(buckets.loc['larga', 'test']),
    'count_threshold': COUNT_THRESHOLD,
    'vocab_en': len(eng_word2idx),
    'vocab_es': len(spa_word2idx),
    'test con inglés repetido en train (%)': round(100 * overlap_test / len(test_pairs), 2),
}
with open(os.path.join(DATA_DIR, 'summary_01.json'), 'w') as f:
    json.dump(summary, f, indent=2)
pd.Series(summary, name='valor').to_frame()